In [1]:
import os
import importlib
import train_utils

os.chdir("c:\\Users\\HANJ29\\Applications\\deepstock\\")

from train_utils import (
    print_label_distribution,
    load_train_test_dataset,
    train_model,
    save_model,
    print_fit_report,
    draw_learning_curve,
    plot_histograms,
)

importlib.reload(train_utils)  # 重新加载模块

freq = "W"
version = "0.2"
model_name = "RF"
label = [
    
    "top_or_bottom",
    "top_or_bottom_stat",
    "top_bottom_volatility_stat",
    "top_or_bottom_optimized",
    "top_or_bottom_stat_optimized",
    "top_bottom_volatility_optimized",
]
features = [
    "change",
    "pct_chg",
    "vol",  # 0.02
    "atr",
    "pct_vol_chg",
    "pct_o2c",
    "lower_shadow",
    "upper_shadow",
    "dif",  # 0.02
    "dea",  # 0.02
    "bar",
    "rsi_6",
    "rsi_12",
    "rsi_24",
    "k",
    "d",
    "j",
    "turnover_rate",
    "turnover_rate_f",
    "volume_ratio",
    # "pe",
    # "pe_ttm", 当为负值时得到的数据为NaN
    "pb",
    "ps",  # 0.02
    "ps_ttm",  # 0.02
    "dv_ratio",  # 0.02
    "dv_ttm",  # 0.02
    "total_share",  # 0.02
    "float_share",  # 0.02
    "free_share",
    "total_mv",  # 0.02
    "circ_mv",  # 0.02
    "float_share_ratio",  # 0.02
    "free_share_ratio",  # 0.02
    "mab_10",
    "mab_25",
    "mab_60",
    "mab_120",
    "mab_200",
]

features_high = [
    "change",
    "pct_chg",
    "vol",  # 0.02
    "atr",
    "pct_vol_chg",
    "pct_o2c",
    "lower_shadow",
    "upper_shadow",
    "dif",  # 0.02
    "dea",  # 0.02
    "bar",
    "rsi_6",
    "rsi_12",
    "rsi_24",
    "k",
    "d",
    "j",
    "turnover_rate",
    "turnover_rate_f",
    "volume_ratio",
    # "pe",
    # "pe_ttm", 当为负值时得到的数据为NaN
    "pb",#!
    "ps",  # 0.02!
    "ps_ttm",  # 0.02
    # "dv_ratio",  # 0.02
    # "dv_ttm",  # 0.02
    "total_share",  # 0.02
    "float_share",  # 0.02 !
    "free_share", #!
    "total_mv",  # 0.02!
    "circ_mv",  # 0.02
    # "float_share_ratio",  ## 0.02!
    "free_share_ratio",  # 0.02
    "mab_10",
    "mab_25",
    "volatility_ratio",# 
    "shadow_ratio",# 
    "mab_60",
    "mab_120",
    "mab_200",
    # "skewness",
    # "kurtosis",
]

c:\Users\HANJ29\Applications\venv-stock-insider-admin\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import pandas as pd

X_train = pd.read_csv("C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/train_dataset_W_0.1.csv")
X_test = pd.read_csv("C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/test_dataset_W_0.1.csv")

In [7]:
# 定义条件字典
conditions = {
    "change": (X_train["change"] >= -40) & (X_train["change"] <= 40),
    "pct_chg": (X_train["pct_chg"] >= -100) & (X_train["pct_chg"] <= 50),
    "vol": (X_train["vol"] >= 0) & (X_train["vol"] <= 1000000000),
    "atr": (X_train["atr"] >= 0) & (X_train["atr"] <= 8),
    "pct_vol_chg": (X_train["pct_vol_chg"] >= -100) & (X_train["pct_vol_chg"] <= 50),
    "pct_o2c": (X_train["pct_o2c"] >= -0.3) & (X_train["pct_o2c"] <= 0.3),
    "lower_shadow": (X_train["lower_shadow"] >= 0) & (X_train["lower_shadow"] <= 0.12),
    "upper_shadow": (X_train["upper_shadow"] >= 0) & (X_train["upper_shadow"] <= 0.15),
    "dif": (X_train["dif"] >= -20) & (X_train["dif"] <= 20),
    "dea": (X_train["dea"] >= -20) & (X_train["dea"] <= 20),
    "bar": (X_train["bar"] >= -5) & (X_train["bar"] <= 5),
    "rsi_6": (X_train["rsi_6"] >= 5) & (X_train["rsi_6"] <= 95),
    "rsi_12": (X_train["rsi_12"] >= 5) & (X_train["rsi_12"] <= 95),
    "rsi_24": (X_train["rsi_24"] >= 5) & (X_train["rsi_24"] <= 95),
    "k": (X_train["k"] >= 0) & (X_train["k"] <= 95),
    "d": (X_train["d"] >= 0) & (X_train["d"] <= 95),
    "j": (X_train["j"] >= -40) & (X_train["j"] <= 140),
    "turnover_rate": (X_train["turnover_rate"] >= 0) & (X_train["turnover_rate"] <= 80),
    "turnover_rate_f": (X_train["turnover_rate_f"] >= 0) & (X_train["turnover_rate_f"] <= 8000),
    "volume_ratio": (X_train["volume_ratio"] >= 0) & (X_train["volume_ratio"] <= 90),
    "pb": (X_train["pb"] >= 0) & (X_train["pb"] <= 1000),
    # "pe": (X_train["pe"] >= 0) & (X_train["pe"] <= 3000),
    # "pe_ttm": (X_train["pe_ttm"] >= 0) & (X_train["pe_ttm"] <= 200000),
    "ps": (X_train["ps"] >= 0) & (X_train["ps"] <= 20000),
    "ps_ttm": (X_train["ps_ttm"] >= 0) & (X_train["ps_ttm"] <= 20000),
    "dv_ratio": (X_train["dv_ratio"] >= 0) & (X_train["dv_ratio"] <= 4),
    "dv_ttm": (X_train["dv_ttm"] >= 0) & (X_train["dv_ttm"] <= 6),
    "total_share": (X_train["total_share"] >= 0) & (X_train["total_share"] <= 2000000),
    "float_share": (X_train["float_share"] >= 0) & (X_train["float_share"] <= 2000000),
    "free_share": (X_train["free_share"] >= 0) & (X_train["free_share"] <= 300000),
    "total_mv": (X_train["total_mv"] >= 0) & (X_train["total_mv"] <= 20000000),
    "circ_mv": (X_train["circ_mv"] >= 0) & (X_train["circ_mv"] <= 20000000),
    # "float_share_ratio": (X_train["float_share_ratio"] >= 0.1) & (X_train["float_share_ratio"] <= 1),
    # "free_share_ratio": (X_train["free_share_ratio"] >= 0.15) & (X_train["free_share_ratio"] <= 0.85),
    "mab_10": (X_train["mab_10"] >= -0.6) & (X_train["mab_10"] <= 0.6),
    "mab_25": (X_train["mab_25"] >= -0.8) & (X_train["mab_25"] <= 0.8),
    "mab_60": (X_train["mab_60"] >= -1.2) & (X_train["mab_60"] <= 0.6),
    "mab_120": (X_train["mab_120"] >= -1.6) & (X_train["mab_120"] <= 2.4),
    "mab_200": (X_train["mab_200"] >= -0.8) & (X_train["mab_200"] <= 2.4),
}

In [9]:
# 将所有条件组合
combined_condition = True
for cond in conditions.values():
    combined_condition &= cond

# 应用条件过滤 X_train
X_train = X_train[combined_condition]
X_train.shape

(1511668, 64)

In [10]:
# 应用条件过滤 X_train
X_test= X_test[combined_condition]
X_test.shape

C:\Users\HANJ29\AppData\Local\Temp\ipykernel_18220\3803768030.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_test= X_test[combined_condition]


(378182, 64)

In [ ]:
# X_train.to_csv("C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/train_dataset_W_0.2.csv", index=False)
# X_test.to_csv("C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/test_dataset_W_0.2.csv", index=False)

In [2]:
# version = "0.2"
X_train, X_test, y_train, y_test = load_train_test_dataset(
    freq=freq,
    version=version,
    features=features,
)

Train dataset loaded from C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training\train_dataset_W_0.2.csv
Test dataset loaded from C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training\test_dataset_W_0.2.csv
split train data to X and y
X_train shape: (1511668, 37), y_train shape: 6
split test data to X and y
X_test shape: (378182, 37), y_test shape: 6


模型code
- LR - logistic regression
- RF - random forest
- SVM - support vector machine
- ANN - artificial neural network
- DT - decision tree
- GBM - gradient boosting machine
- XGB - extreme gradient boosting
- LGB - light gradient boosting
- CAT - catboost


In [30]:
# 训练模型
# Current label distribution:
# top_or_bottom_optimized
# 0    268571
# 2     72591
# 1     72369
# 定义超参数字典
def get_hyper_param(param_name):
    RF = {
        "n_estimators": 100,  # 初始树的数量，适中即可，后续可增加
        "max_depth": None,  # 不限制树的深度，观察模型是否过拟合
        "min_samples_split": 10,  # 20,#10,      # 内部节点再分裂所需的最小样本数
        "min_samples_leaf": 5,  # 10,#5,        # 叶子节点的最小样本数，防止过拟合
        "max_features": "sqrt",  # 每次分裂时考虑的最大特征数，sqrt 是推荐值
        "class_weight": "balanced",  # 根据类别样本数量自动调整权重
        "random_state": 42,  # 固定随机种子，保证结果可复现
        "n_jobs": -1,  # 使用所有可用 CPU 核心进行并行计算
        "verbose": 1,  # 输出详细信息
    }
    # 定义超参数字典
    XGB = {
        "n_estimators": 5000,  # 如果过拟合，可以适当增加
        "learning_rate": 0.01,  # 如果过拟合，可以适当减小；反之亦然
        "max_depth": 10,  # 如果过拟合，可以适当减小；反之亦然
        "min_child_weight": 7,  # 如果过拟合，可以适当增大；反之亦然
        "gamma": 1,  # 如果过拟合，可以适当增大；反之亦然
        "subsample": 0.8,  # 如果过拟合，可以适当减小；反之亦然
        "colsample_bytree": 0.8,  # 如果过拟合，可以适当减小；反之亦然
        "reg_alpha": 15,  # 如果过拟合，可以适当增大；反之亦然
        "reg_lambda": 25,  # 如果过拟合，可以适当增大；反之亦然
        "objective": "multi:softmax",  # 多分类
        "num_class": 3,  # 类别数
        "eval_metric": "mlogloss",  # 评价指标
        # "early_stopping_rounds": 50, #早停轮数
        # "use_label_encoder": False,  # 避免警告
        "random_state": 42,  # 随机种子
        "n_jobs": -1,  # 使用所有可用 CPU 核心进行并行计算
        "verbosity": 1,  # 输出详细信息
    }
    # 超参数
    LGBM = {
        "n_estimators": 10000,  # 增加迭代次数
        "learning_rate": 0.01,  # 减小学习率
        "max_depth": 4,  # 降低树的深度，避免过拟合
        "num_leaves": 31,  # 控制叶子节点数量，避免过拟合
        "min_child_samples": 20,  # 增加叶子节点的最小样本数
        "min_split_gain": 0.1,  # 增加分裂增益阈值
        "subsample": 0.8,  # 减少样本采样比例
        "colsample_bytree": 0.8,  # 减少特征采样比例
        "reg_alpha": 1,  # 增加 L1 正则化
        "reg_lambda": 10,  # 增加 L2 正则化
        "objective": "multiclass",  # 多分类任务
        "num_class": 3,  # 类别数
        "metric": "multi_logloss",  # 多分类损失
        "random_state": 42,  # 随机种子
    }
    # 定义超参数字典
    CAT = {
        "iterations": 10000,  # 增加迭代次数
        "learning_rate": 0.01,  # 减小学习率
        "depth": 6,  # 降低树的深度，避免过拟合
        "l2_leaf_reg": 10,  # 增加 L2 正则化
        "bagging_temperature": 1,  # 调整 Bagging 温度
        "random_strength": 1,  # 增加随机分裂强度
        "loss_function": "MultiClass",  # 多分类任务
        "eval_metric": "MultiClass",  # 多分类损失
        "verbose": 100,  # 每隔 100 次迭代打印日志
        "random_seed": 42,  # 随机种子
    }
    local_vars = locals()
    return local_vars[param_name] if param_name in local_vars else None

In [83]:
label = 'top_bottom_volatility_optimized'

In [84]:
print_label_distribution(y_train[label])

Current label distribution:
top_bottom_volatility_optimized
0    1438381
1      37475
2      35812
Name: count, dtype: int64


In [ ]:
# 示例调用
# 假设 X_train 是你的数据集，features 是特征列表
plot_histograms(X_train, features)

In [86]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# 获取每个类别的数量
COUNT_PER_CLASS = 100000 # 每个类别的样本数量
class_counts = y_train[label].value_counts()
print("原始类别分布:")
print(class_counts)

# 确定少数类的数量
minority_class_counts = class_counts.min()

# sampling_strategy = {0: minority_class_counts}  # 类别 0 保持 1000，类别 1 和 2 过采样到 500
sampling_strategy = {0: int(minority_class_counts * 2)}
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train[label])
y_resampled.value_counts()

原始类别分布:
top_bottom_volatility_optimized
0    1438381
1      37475
2      35812
Name: count, dtype: int64


top_bottom_volatility_optimized
0    71624
1    37475
2    35812
Name: count, dtype: int64

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

sampling_strategy = {1: int(minority_class_counts * 1.5), 2: int(minority_class_counts * 1.5)}  # 类别 0 保持 1000，类别 1 和 2 过采样到 500
smote = SMOTEENN(sampling_strategy=sampling_strategy, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_resampled, y_resampled)
y_resampled.value_counts()

c:\Users\HANJ29\Applications\venv-stock-insider-admin\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\HANJ29\Applications\venv-stock-insider-admin\lib\site-packages\joblib\externals\loky\backend\context.py", line 282, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [ ]:
# X_resampled.to_csv("C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/train_dataset_W_0.2_resampled.csv", index=False)
# y_resampled.to_csv("C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/train_labels_W_0.2_resampled.csv", index=False)

In [ ]:
# X_resampled = pd.read_csv("C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/train_dataset_W_0.2_resampled.csv")
# y_resampled = pd.read_csv("C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/dataset_training/train_labels_W_0.2_resampled.csv")

In [87]:
model_name = "XGB"
params = get_hyper_param(model_name)
params

{'n_estimators': 5000,
 'learning_rate': 0.01,
 'max_depth': 10,
 'min_child_weight': 7,
 'gamma': 1,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'reg_alpha': 15,
 'reg_lambda': 25,
 'objective': 'multi:softmax',
 'num_class': 3,
 'eval_metric': 'mlogloss',
 'random_state': 42,
 'n_jobs': -1,
 'verbosity': 1}

In [11]:
X_train['volatility_ratio'] = X_train['pct_vol_chg'] / (X_train['atr'] + 1e-5)
X_train['shadow_ratio'] = X_train['lower_shadow'] / (X_train['upper_shadow'] + 1e-5)

In [92]:
X_train_high = X_train[features_high]

In [88]:
X_resampled['volatility_ratio'] = X_resampled['pct_vol_chg'] / (X_resampled['atr'] + 1e-5)
X_resampled['shadow_ratio'] = X_resampled['lower_shadow'] / (X_resampled['upper_shadow'] + 1e-5)
X_resampled_high = X_resampled[features_high]

In [89]:
X_resampled_high.columns

Index(['change', 'pct_chg', 'vol', 'atr', 'pct_vol_chg', 'pct_o2c',
       'lower_shadow', 'upper_shadow', 'dif', 'dea', 'bar', 'rsi_6', 'rsi_12',
       'rsi_24', 'k', 'd', 'j', 'turnover_rate', 'turnover_rate_f',
       'volume_ratio', 'pb', 'ps', 'ps_ttm', 'total_share', 'float_share',
       'free_share', 'total_mv', 'circ_mv', 'free_share_ratio', 'mab_10',
       'mab_25', 'volatility_ratio', 'shadow_ratio', 'mab_60', 'mab_120',
       'mab_200'],
      dtype='object')

In [ ]:
# 训练模型
model = train_model(
    # X_train, y_train["top_or_bottom_optimized"], model_name="XGB", **params_xgb  # with_learning_curve=True
    X_train,
    y_train[label],
    model_name=model_name,
    **params  # with_learning_curve=True
)
print("Model training complete.")

In [90]:
from datetime import datetime

start_time = datetime.now()
print(f"Starting model training...{start_time}")
# 训练模型
model = train_model(
    X_resampled_high,
    y_resampled,
    model_name=model_name,  # with_learning_curve=True
    **params,  # with_learning_curve=True
    # sample_weight=sample_weights,
)
print("Model training complete.")
end_time = datetime.now()
time_diff = end_time - start_time
print(f"Model training time: {time_diff}")
print("Model training complete.")

Starting model training...2025-05-05 15:11:11.014136
Model training complete.
Model training time: 0:17:36.336543
Model training complete.


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score( model, X_train, y_train[label], cv=5, scoring='recall_macro')
print("Cross-validated recall:", scores.mean())

In [93]:
print("Trainig report:")
report_train = print_fit_report(model, X_train_high, y_train[label])

Trainig report:
Recall (weighted): 0.7383
              precision    recall  f1-score   support

           0       0.98      0.74      0.84   1438381
           1       0.13      0.72      0.22     37475
           2       0.12      0.74      0.20     35812

    accuracy                           0.74   1511668
   macro avg       0.41      0.73      0.42   1511668
weighted avg       0.94      0.74      0.81   1511668



In [91]:
print("Trainig report:")
report_train = print_fit_report(model, X_resampled_high, y_resampled)

Trainig report:
Recall (weighted): 0.7752
              precision    recall  f1-score   support

           0       0.76      0.82      0.79     71624
           1       0.81      0.72      0.76     37475
           2       0.78      0.74      0.76     35812

    accuracy                           0.78    144911
   macro avg       0.78      0.76      0.77    144911
weighted avg       0.78      0.78      0.77    144911



In [16]:
X_test['volatility_ratio'] = X_test['pct_vol_chg'] / (X_test['atr'] + 1e-5)
X_test['shadow_ratio'] = X_test['lower_shadow'] / (X_test['upper_shadow'] + 1e-5)
# X_test['skewness'] = X_test['pct_chg'].skew()
# X_test['kurtosis'] = X_test['pct_chg'].kurt()


In [17]:
X_test_high = X_test[features_high]

In [ ]:
from xgboost import plot_importance
plot_importance(model)

In [98]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# 获取每个类别的数量
COUNT_PER_CLASS = 100000 # 每个类别的样本数量
class_counts = y_test[label].value_counts()
print("原始类别分布:")
print(class_counts)

# 确定少数类的数量
minority_class_counts = class_counts.min()

# sampling_strategy = {0: minority_class_counts}  # 类别 0 保持 1000，类别 1 和 2 过采样到 500
sampling_strategy = {0: int(minority_class_counts * 1.5)}
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_test_resampled, y_test_resampled = rus.fit_resample(X_test_high, y_test[label])

原始类别分布:
top_bottom_volatility_optimized
0    359714
1     10684
2      7784
Name: count, dtype: int64


In [96]:
y_test_resampled.value_counts()

top_bottom_volatility_optimized
0    15568
1    10684
2     7784
Name: count, dtype: int64

In [41]:
X_test_resampled.columns

Index(['change', 'pct_chg', 'vol', 'atr', 'pct_vol_chg', 'pct_o2c',
       'lower_shadow', 'upper_shadow', 'dif', 'dea', 'bar', 'rsi_6', 'rsi_12',
       'rsi_24', 'k', 'd', 'j', 'turnover_rate', 'turnover_rate_f',
       'volume_ratio', 'pb', 'ps', 'ps_ttm', 'total_share', 'float_share',
       'free_share', 'total_mv', 'circ_mv', 'free_share_ratio', 'mab_10',
       'mab_25', 'volatility_ratio', 'shadow_ratio', 'mab_60', 'mab_120',
       'mab_200'],
      dtype='object')

In [99]:
print("Test dataset split into features and labels.")
report_test = print_fit_report(model, X_test_resampled, y_test_resampled)

Test dataset split into features and labels.
Recall (weighted): 0.5839
              precision    recall  f1-score   support

           0       0.49      0.74      0.59     11676
           1       0.76      0.40      0.52     10684
           2       0.69      0.60      0.64      7784

    accuracy                           0.58     30144
   macro avg       0.65      0.58      0.58     30144
weighted avg       0.64      0.58      0.58     30144



In [100]:
from train_utils import save_training_results

save_training_results(
    model_name,
    params,
    report_train,
    report_test,
    version,
    freq,
    y_train[label].value_counts(),
    output_file="training_results_W.csv",
)

Results saved to training_results_W.csv


In [ ]:
print("Start Drawing Learning Curve")
draw_learning_curve(model, X_train, y_train, cv=5, scoring=None, title="Learning Curve")

In [101]:
# 保存模型
save_model(model, model_name, freq=freq, pred_type="VOLOPT")
print("Model saved.")

Model saved successfully to C:/Users/HANJ29/Applications/btweb/stock_filestore/DEV/models\XGB_VOLOPT_W.model
Model saved.


dataset内容分析和重新采样

处理周线/月线训练集重新采样